# Незалежна перевірка запущеного HTTP API
**Лабораторна робота 1. Частина 3: Незалежна HTTP-перевірка**  
**Об'єкт тестування:** Task Registry Service (`http://127.0.0.1:8000`)  
**Інструмент:** Зовнішній HTTP-клієнт на базі `urllib` (чорна скринька)  

Notebook не імпортує код застосунку, не читає базу даних напряму й взаємодіє із сервісом виключно через мережевий HTTP API.

## 1. Імпорти

Використовуємо стандартну бібліотеку Python: `urllib.request` та `urllib.error`.

In [1]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from IPython.display import Markdown, display

print("Клієнтські бібліотеки імпортовано.")

Клієнтські бібліотеки імпортовано.


## 2. Адреса й контракт

Сервер має бути запущений:  
`uv run uvicorn app.main:app --port 8000`  
Робоча тека: корінь проєкту `task-registry-service`.

In [2]:
BASE_URL = "http://127.0.0.1:8000"
LIST_PATH = "/tasks"
MANUAL_PATH = "/tasks"
AI_PATH = "/tasks/from-text"
HTTP_TIMEOUT_SECONDS = 30
LIST_HTTP = 200
MANUAL_HTTP = 201
INVALID_MANUAL_HTTP = 422
CASE_RESULTS = []

## 3. Функція виклику API

Надсилає HTTP-запит, перевіряє коректність JSON та повертає статус і тіло.

In [3]:
def call_api(method, path, payload=None):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = Request(
        BASE_URL.rstrip("/") + path,
        data=data,
        headers={"Accept": "application/json", "Content-Type": "application/json"},
        method=method,
    )
    try:
        with urlopen(request, timeout=HTTP_TIMEOUT_SECONDS) as response:
            status, raw = response.status, response.read()
    except HTTPError as error:
        with error:
            status, raw = error.code, error.read()
    except (URLError, TimeoutError, OSError) as error:
        raise RuntimeError("HTTP-запит не завершився. Перевірте сервер, адресу й порт.") from error
    try:
        body = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError(f"HTTP {status}: відповідь не відповідає JSON-контракту: {raw}") from error
    print(f"{method} {path} → HTTP {status}")
    return status, body

## 4. Допоміжні функції реєстрації та знімка стану сховища

In [4]:
def run_check(name, check):
    try:
        check()
    except Exception as error:
        row = {"scenario": name, "status": "failed", "reason": f"{type(error).__name__}: {error}"}
    else:
        row = {"scenario": name, "status": "passed", "reason": "Усі задані перевірки пройдено"}
    CASE_RESULTS.append(row)
    print(f"[{row['status'].upper()}] {row['scenario']}: {row['reason']}")


def get_records():
    status, body = call_api("GET", LIST_PATH)
    assert status == LIST_HTTP, f"Очікували HTTP {LIST_HTTP}, отримали {status}"
    assert isinstance(body, list), "Список має бути JSON-масивом"
    assert all(isinstance(record, dict) and "id" in record for record in body)
    return body


def snapshot(records):
    ids = [record["id"] for record in records]
    assert len(ids) == len(set(ids)), "Ідентифікатори записів мають бути унікальними"
    return {record["id"]: record for record in records}

## Сценарій 1. Список і початковий стан (AC-1)

In [5]:
def check_initial_state():
    records = get_records()
    print(f"Початкова кількість записів: {len(records)}")
    assert isinstance(records, list)

run_check("Список і початковий стан", check_initial_state)

GET /tasks → HTTP 200
Початкова кількість записів: 3
[PASSED] Список і початковий стан: Усі задані перевірки пройдено


## Сценарій 2. Ручне створення (AC-2)

In [6]:
MANUAL_INPUT = {
    "title": "Підготувати звіт",
    "description": "Звіт тестування",
    "assignee": "Ірина",
    "due_date": "2026-11-20"
}

def check_manual_create():
    before = snapshot(get_records())
    status, created = call_api("POST", MANUAL_PATH, MANUAL_INPUT)
    assert status == MANUAL_HTTP, f"Очікували HTTP {MANUAL_HTTP}, отримали {status}"
    assert isinstance(created, dict) and "id" in created, "Відповідь не містить id"
    for key, val in MANUAL_INPUT.items():
        assert created.get(key) == val, f"Невідповідність поля {key}"
    assert created["id"] not in before, "Новий ID вже існував у сховищі"
    after = snapshot(get_records())
    assert len(after) == len(before) + 1, "Кількість записів не зросла на 1"
    assert created["id"] in after
    assert after[created["id"]] == created
    assert all(after[k] == v for k, v in before.items()), "Попередні записи були пошкоджені!"

run_check("Ручне створення", check_manual_create)

GET /tasks → HTTP 200
POST /tasks → HTTP 201
GET /tasks → HTTP 200
[PASSED] Ручне створення: Усі задані перевірки пройдено


## Сценарій 3. Хибний ручний вхід (AC-3)

In [7]:
INVALID_MANUAL_INPUT = {
    "title": "X",  # Менше мінімальної довжини 3
    "description": "Тест",
    "assignee": "Ірина",
    "due_date": "не-дата"
}

def check_invalid_manual():
    before = snapshot(get_records())
    status, error_body = call_api("POST", MANUAL_PATH, INVALID_MANUAL_INPUT)
    assert status == INVALID_MANUAL_HTTP, f"Очікували {INVALID_MANUAL_HTTP}, отримали {status}"
    after = snapshot(get_records())
    assert after == before, "Сховище було змінено після невалідного запиту!"

run_check("Хибний ручний вхід", check_invalid_manual)

GET /tasks → HTTP 200
POST /tasks → HTTP 422
GET /tasks → HTTP 200
[PASSED] Хибний ручний вхід: Усі задані перевірки пройдено


## Сценарій 4. Коректна AI-пропозиція (AC-4)

In [8]:
AI_VALID_INPUT = {
    "text": "Підготувати релізний звіт; опис: перевірити покриття коду; відповідальний: Максим; строк: 2026-11-25"
}

def check_ai_valid():
    before = snapshot(get_records())
    status, created = call_api("POST", AI_PATH, AI_VALID_INPUT)
    assert status == 201, f"Очікували HTTP 201, отримали {status}"
    assert created["assignee"] == "Максим"
    assert created["due_date"] == "2026-11-25"
    assert created["status"] == "created"
    after = snapshot(get_records())
    assert len(after) == len(before) + 1
    assert created["id"] in after
    assert all(after[k] == v for k, v in before.items())

run_check("Коректна AI-пропозиція", check_ai_valid)

GET /tasks → HTTP 200
POST /tasks/from-text → HTTP 201
GET /tasks → HTTP 200
[PASSED] Коректна AI-пропозиція: Усі задані перевірки пройдено


## Сценарій 5. Неповний або неоднозначний текст (AC-4)

In [9]:
AI_INCOMPLETE_INPUT = {
    "text": "Оновити залежності бекенду; опис: перевірити сумісність; відповідальний: Олексій"
}

def check_ai_incomplete():
    before = snapshot(get_records())
    status, body = call_api("POST", AI_PATH, AI_INCOMPLETE_INPUT)
    assert status == 422, f"Очікували HTTP 422, отримали {status}"
    assert body.get("status") == "needs_clarification"
    after = snapshot(get_records())
    assert after == before, "Сховище не повинно змінюватися при неповному тексті!"

run_check("Неповний або неоднозначний текст", check_ai_incomplete)

GET /tasks → HTTP 200
POST /tasks/from-text → HTTP 422
GET /tasks → HTTP 200
[PASSED] Неповний або неоднозначний текст: Усі задані перевірки пройдено


## Сценарій 6. Структурно хибна пропозиція (AC-4)

In [10]:
AI_MALFORMED_INPUT = {
    "text": "Оновити сервіс; simulate_malformed_json; відповідальний: Ірина; строк: 2026-11-20"
}

def check_ai_malformed():
    before = snapshot(get_records())
    status, body = call_api("POST", AI_PATH, AI_MALFORMED_INPUT)
    assert status == 422, f"Очікували HTTP 422, отримали {status}"
    assert body.get("status") == "invalid_semantics"
    after = snapshot(get_records())
    assert after == before, "Сховище не повинно змінюватися при структурній помилці!"

run_check("Структурно хибна пропозиція", check_ai_malformed)

GET /tasks → HTTP 200
POST /tasks/from-text → HTTP 422
GET /tasks → HTTP 200
[PASSED] Структурно хибна пропозиція: Усі задані перевірки пройдено


## Сценарій 7. Змістовно хибна пропозиція / Галюцинація (AC-4)

In [11]:
AI_HALLUCINATION_INPUT = {
    "text": "Узгодити контракт із постачальником; опис: підписати договір; відповідальний: Ірина; строк: 2026-11-30"
}

def check_ai_hallucination():
    before = snapshot(get_records())
    status, body = call_api("POST", AI_PATH, AI_HALLUCINATION_INPUT)
    assert status == 422, f"Очікували HTTP 422, отримали {status}"
    assert body.get("status") == "invalid_semantics"
    assert "галюцинація" in body.get("reason", "").lower()
    after = snapshot(get_records())
    assert after == before, "Сховище не повинно змінюватися при галюцинації!"

run_check("Змістовно хибна пропозиція", check_ai_hallucination)

GET /tasks → HTTP 200


POST /tasks/from-text → HTTP 422
GET /tasks → HTTP 200
[PASSED] Змістовно хибна пропозиція: Усі задані перевірки пройдено


## Сценарій 8. Збій моделі й незалежний ручний шлях (AC-5)

In [12]:
def check_provider_failure_and_resilience():
    before = snapshot(get_records())
    # 1. Запит до AI з помилкою провайдера
    status, body = call_api("POST", AI_PATH, {"text": "simulate_provider_error: термінова задача"})
    assert status == 503, f"Очікували HTTP 503, отримали {status}"
    assert body.get("status") == "provider_unavailable"
    assert snapshot(get_records()) == before, "Сховище не повинно змінюватися при 503!"

    # 2. Перевіряємо, що ручний шлях продовжує працювати безперебійно
    m_status, m_created = call_api("POST", MANUAL_PATH, {
        "title": "Ручна задача після збою",
        "description": "Перевірка незалежності",
        "assignee": "Тарас",
        "due_date": "2026-11-28"
    })
    assert m_status == 201
    after = snapshot(get_records())
    assert len(after) == len(before) + 1
    assert m_created["id"] in after

run_check("Збій моделі й незалежний ручний шлях", check_provider_failure_and_resilience)

GET /tasks → HTTP 200
POST /tasks/from-text → HTTP 503
GET /tasks → HTTP 200
POST /tasks → HTTP 201
GET /tasks → HTTP 200
[PASSED] Збій моделі й незалежний ручний шлях: Усі задані перевірки пройдено


## Сценарій 9. Жива інтеграція через сервіс

Перевірка живого виклику до OpenAI / OpenRouter. Якщо ключ не встановлено — чесний статус `not_run`.

In [13]:
import os

def check_live_integration():
    api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key:
        row = {
            "scenario": "Жива інтеграція через сервіс",
            "status": "not_run",
            "reason": "API ключ не встановлено в середовищі. Детерміновані сценарії успішно перевірені."
        }
        CASE_RESULTS.append(row)
        print(f"[{row['status'].upper()}] {row['scenario']}: {row['reason']}")
        return

    # Якщо ключ є - викликаємо живий сценарій
    status, created = call_api("POST", AI_PATH, {
        "text": "Провести демонстрацію релізу; відповідальний: Ірина; строк: 2026-11-29"
    })
    assert status in [201, 422, 503]
    row = {
        "scenario": "Жива інтеграція через сервіс",
        "status": "passed",
        "reason": f"Живий виклик повернув HTTP {status}"
    }
    CASE_RESULTS.append(row)
    print(f"[{row['status'].upper()}] {row['scenario']}: {row['reason']}")

check_live_integration()

[NOT_RUN] Жива інтеграція через сервіс: API ключ не встановлено в середовищі. Детерміновані сценарії успішно перевірені.


## 8. Підсумок і матриця перевірки

In [14]:
required = [
    "Список і початковий стан",
    "Ручне створення",
    "Хибний ручний вхід",
    "Коректна AI-пропозиція",
    "Неповний або неоднозначний текст",
    "Структурно хибна пропозиція",
    "Змістовно хибна пропозиція",
    "Збій моделі й незалежний ручний шлях",
    "Жива інтеграція через сервіс",
]

latest = {row["scenario"]: row for row in CASE_RESULTS}
rows = [latest.get(name, {"scenario": name, "status": "not_run", "reason": "Немає результату"}) for name in required]
table = "| Сценарій | Результат | Причина |\n|---|---|---|\n"
for row in rows:
    reason = str(row["reason"]).replace("|", "/").replace("\n", " ")
    table += f"| {row['scenario']} | `{row['status']}` | {reason} |\n"
display(Markdown(table))

for status in ("passed", "failed", "not_run"):
    print(f"{status}: {sum(row['status'] == status for row in rows)}")

failed_count = sum(row['status'] == 'failed' for row in rows)
assert failed_count == 0, "Жоден обов'язковий сценарій не повинен мати статус failed!"
print("Перевірка завершена успішно: усі обов'язкові детерміновані тести PASSED.")

| Сценарій | Результат | Причина |
|---|---|---|
| Список і початковий стан | `passed` | Усі задані перевірки пройдено |
| Ручне створення | `passed` | Усі задані перевірки пройдено |
| Хибний ручний вхід | `passed` | Усі задані перевірки пройдено |
| Коректна AI-пропозиція | `passed` | Усі задані перевірки пройдено |
| Неповний або неоднозначний текст | `passed` | Усі задані перевірки пройдено |
| Структурно хибна пропозиція | `passed` | Усі задані перевірки пройдено |
| Змістовно хибна пропозиція | `passed` | Усі задані перевірки пройдено |
| Збій моделі й незалежний ручний шлях | `passed` | Усі задані перевірки пройдено |
| Жива інтеграція через сервіс | `not_run` | API ключ не встановлено в середовищі. Детерміновані сценарії успішно перевірені. |


passed: 8
failed: 0
not_run: 1
Перевірка завершена успішно: усі обов'язкові детерміновані тести PASSED.


## Самоперевірка й висновок

1. **Що довела відповідь сервера, а що довело порівняння зі специфікацією?**  
   - Відповідь сервера довела коректність HTTP-кодів (200, 201, 422, 503) та структуру поверненого JSON.  
   - Порівняння зі специфікацією довело, що дані потрапляють у сховище лише при виконанні всіх бізнес-правил (Grounding Check, непорожність, формат дати), а відхилені запити залишають стан сховища незмінним.

2. **Чому незмінної кількості записів недостатньо для перевірки відмови?**  
   - Якби ми перевіряли лише `len(after) == len(before)`, ми б не помітили баг, за якого один запис випадково перезаписав або видалив інший запис (заміна даних). Функція `snapshot()` перевіряє рівність повного словника `{id: record}`.

3. **Який сценарій виявив проблему і що змінилося після виправлення?**  
   - Сценарій перевірки чутливості до галюцинацій (мутаційний тест) продемонстрував, що без правила `assignee in raw_text` модель здатна приписувати задачі третім особам (наприклад, призначити Богдана замість Ірини). Додавання обов'язкового Grounding Check у `policy.py` усунуло цей дефект.

4. **Які результати отримані з підготовленими відповідями, а які з реальною моделлю?**  
   - Сценарії 1–8 виконано детерміновано за допомогою `MockProposer` із контрольованими входами та збоями, що гарантує 100% повторюваність перевірки в CI без витрат.  
   - Сценарій 9 підтримує живий виклик реальної моделі або чесно маркується як `not_run` за відсутності ключа.